# Set up

Download August 2026 data to then load it to the database
```bash
# sync the libraries installed
pipenv shell
pipenv install -d Pipfile # to install all missing dependencies

# Data from August 2026 
scraper update 2026-08-01 2026-08-31
```

In [1]:
import ast
import sqlite3
from datetime import datetime
from pathlib import Path

import pandas as pd
import ollama
from tqdm import tqdm

from un_llm_analysis.extractor import extract_briefing, ExtractedSection, ExtractedBriefing
from un_llm_analysis.config import RAW_DATA_DIR
from un_llm_analysis.utils import SCHEMA_DB, BRIEFINGS_DB
from un_llm_analysis.database import (
    insert_into_briefing,
    insert_into_section,
    create_database,
    get_db_connection,
    get_briefing_id_from_date,
    get_sections_by_briefing_id,
    get_briefings_between_dates,
    insert_into_sdg,
    delete_from_sdg,
)

In [2]:
# Create database
create_database()

2026-09-17 00:30:05.953 | INFO     | un_llm_analysis.database:create_database:25 - Database created in "/Users/darenasc/projects/correlaid/correlaid-mentoring-project/app/briefings.db"


In [ ]:
# def get_sdgs_between_dates(
#     conn: sqlite3.Connection, start_date: str, end_date: str
# ) -> pd.DataFrame:
#     """Return data from the sdg table.

#     Args:
#         conn (sqlite3.Connection): Database connection.
#         start_date (str): Start date with format 'YYYY-MM-DD'.
#         end_date (str): End date with format 'YYYY-MM-DD'.

#     Returns:
#         pd.DataFrame: Data from the sdg table.
#     """ """"""
#     min_date = min(get_briefing_id_from_date(start_date), get_briefing_id_from_date(end_date))
#     max_date = max(get_briefing_id_from_date(start_date), get_briefing_id_from_date(end_date))

#     sql_query = """SELECT briefing_id, section_id, sdg, model, created_at FROM sdg WHERE briefing_id between ? AND ?;"""

#     with conn:
#         cursor = conn.cursor()
#         cursor.execute(
#             sql_query,
#             (
#                 min_date,
#                 max_date,
#             ),
#         )
#         rows = cursor.fetchall()
#         column_names = [x[0] for x in cursor.description]
#         cursor.close()
#     df = pd.DataFrame(rows, columns=column_names)
#     return df

In [4]:
# Get a database connection
conn = get_db_connection()

# Load data to the database for this notebook
path = RAW_DATA_DIR / "briefings" / "2026" / "2026-08"
files = list(path.glob("**/*.json"))
for file_path in tqdm(files, total=len(files)):
    briefing = extract_briefing(file_path=file_path)
    insert_into_briefing(conn=conn, briefing=briefing)
    insert_into_section(
        conn=conn,
        briefing_id=get_briefing_id_from_date(briefing.briefing_date),
        sections=briefing.sections,
    )

conn.close()

  0%|          | 0/31 [00:00<?, ?it/s]2026-09-17 00:30:17.073 | INFO     | un_llm_analysis.database:insert_into_briefing:105 - Briefing from '2026-08-20' inserted to the database.
2026-09-17 00:30:17.074 | INFO     | un_llm_analysis.database:delete_sections:479 - Deleted the sections of briefing '20260820'
2026-09-17 00:30:17.077 | INFO     | un_llm_analysis.database:insert_into_briefing:105 - Briefing from '2026-08-16' inserted to the database.
2026-09-17 00:30:17.078 | INFO     | un_llm_analysis.database:delete_sections:479 - Deleted the sections of briefing '20260816'
2026-09-17 00:30:17.081 | INFO     | un_llm_analysis.database:insert_into_briefing:105 - Briefing from '2026-08-01' inserted to the database.
2026-09-17 00:30:17.081 | INFO     | un_llm_analysis.database:delete_sections:479 - Deleted the sections of briefing '20260801'
2026-09-17 00:30:17.095 | INFO     | un_llm_analysis.database:insert_into_briefing:105 - Briefing from '2026-08-17' inserted to the database.
2026-09-17

# SDGs

In [5]:
# https://sdgs.un.org/goals
sdgs = {
    "SDG1": "End poverty in all its forms everywhere",
    "SDG2": "End hunger, achieve food security and improved nutrition and promote sustainable agriculture",
    "SDG3": "Ensure healthy lives and promote well-being for all at all ages",
    "SDG4": "Ensure inclusive and equitable quality education and promote lifelong learning opportunities for all",
    "SDG5": "Achieve gender equality and empower all women and girls",
    "SDG6": "Ensure availability and sustainable management of water and sanitation for all",
    "SDG7": "Ensure access to affordable, reliable, sustainable and modern energy for all",
    "SDG8": "Promote sustained, inclusive and sustainable economic growth, full and productive employment and decent work for all",
    "SDG9": "Build resilient infrastructure, promote inclusive and sustainable industrialization and foster innovation",
    "SDG10": "Reduce inequality within and among countries",
    "SDG11": "Make cities and human settlements inclusive, safe, resilient and sustainable",
    "SDG12": "Ensure sustainable consumption and production patterns",
    "SDG13": "Take urgent action to combat climate change and its impacts",
    "SDG14": "Conserve and sustainably use the oceans, seas and marine resources for sustainable development",
    "SDG15": "Protect, restore and promote sustainable use of terrestrial ecosystems, sustainably manage forests, combat desertification, and halt and reverse land degradation and halt biodiversity loss",
    "SDG16": "Promote peaceful and inclusive societies for sustainable development, provide access to justice for all and build effective, accountable and inclusive institutions at all levels",
    "SDG17": "Strengthen the means of implementation and revitalize the Global Partnership for Sustainable Development",
}

sdg_text = ""
for sdg in sdgs:
    sdg_text += f"* {sdg} {sdgs[sdg]}\n"

sdg_definitions = sdg_text.strip()

In [6]:
promtp = """**Classification Task:**

Classify the following text into one or more of the following 17 Sustainable Development Goals (SDGs):

**Text to be Classified:**

{text}

**SDG Definitions:**

{sdg_definitions}

**Example Response:**

If the text belongs to one or more of the 17 SDGs, provide a list of relevant goal(s) in the format: ['SDG1', 'SDG5'].

**Edge Case:**

If the text does not belong to any of the 17 SDGs, return 'None'. Do not include any text to your answer, just provide the response in a list.
"""

In [15]:
start_date = "2026-08-01"
end_date = "2026-08-07"
conn = get_db_connection()
briefings = get_briefings_between_dates(conn=conn, start_date=start_date, end_date=end_date)

In [14]:
llm = "llama3.2"
conn = get_db_connection()
for briefing in briefings:
    if len(briefing.sections) > 0:
        for i, section in enumerate(briefing.sections, start=1):
            result = ollama.generate(
                model=llm,
                prompt=promtp.format(text=section.text, sdg_definitions=sdg_definitions),
                think=False,
            )

            briefing_id = get_briefing_id_from_date(briefing.briefing_date)

            insert_into_sdg(
                conn=conn,
                briefing_id=briefing_id,
                section_id=briefing_id * 100 + i,
                sdgs=result["response"],
                model=llm,
                created_at=result["created_at"],
            )

conn.close()

2026-09-17 00:38:30.665 | INFO     | un_llm_analysis.database:delete_from_sdg:546 - Deleted sdg section'2026080301' of model 'llama3.2'
2026-09-17 00:38:30.666 | INFO     | un_llm_analysis.database:insert_into_sdg:518 - 2026080301 ['SDG16', 'SDG13'] data inserted into `sdg`
2026-09-17 00:38:31.133 | INFO     | un_llm_analysis.database:delete_from_sdg:546 - Deleted sdg section'2026080302' of model 'llama3.2'
2026-09-17 00:38:31.134 | INFO     | un_llm_analysis.database:insert_into_sdg:518 - 2026080302 ['SDG13', 'SDG16'] data inserted into `sdg`
2026-09-17 00:38:32.007 | INFO     | un_llm_analysis.database:delete_from_sdg:546 - Deleted sdg section'2026080303' of model 'llama3.2'
2026-09-17 00:38:32.008 | INFO     | un_llm_analysis.database:insert_into_sdg:518 - 2026080303 ['SDG3', 'SDG6', 'SDG12', 'SDG13'] data inserted into `sdg`
2026-09-17 00:38:36.782 | INFO     | un_llm_analysis.database:delete_from_sdg:546 - Deleted sdg section'2026080304' of model 'llama3.2'
2026-09-17 00:38:36.783

# Visualisation

In [16]:
import plotly.express as px

conn = get_db_connection()
df = get_sdgs_between_dates(conn=conn, start_date=start_date, end_date=end_date)
x = df[df["model"] == llm][["sdg"]].value_counts().values  # frequency of each SDG in the sections.
y = [
    x[0] for x in df[df["model"] == llm][["sdg"]].value_counts().index
]  # values from the "sdg" column with values: ["SDG1", "SDG2", ..., "SDG17"]

fig = px.bar(
    df,
    x=x,
    y=y,
    orientation="h",
    title=f"17SDGs in {len(df.briefing_id.unique())} Daily Briefings between {str(datetime.strptime(str(df.briefing_id.min()), "%Y%m%d"))[:10]} and {str(datetime.strptime(str(df.briefing_id.max()), "%Y%m%d"))[:10]}",
)
fig.show()

# mlflow (optional)

This is to compare results from different LLMs.

Install the following libraries and dependencies:
```bash
# Download LLMs to test
ollama pull llama3.2
ollama pull gemma3:4b

# Install dependencies to use mlflow with local Ollama
pipenv shell
pipenv install mlflow openai jupyter ipywidgets # or `pipenv install -d Pipfile to sync dependencies`

# Start a local mlflow server
mlflow server
```

In [ ]:
import mlflow
from openai import OpenAI

mlflow.openai.autolog()
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("SDG")

client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="dummy",
)

In [ ]:
models = ["llama3.2:latest", "gemma3:4b"]
for llm in models:
    for i, section in enumerate(df_sections.text.to_list()[:5]):
        response = client.chat.completions.create(
            model=llm,
            messages=[
                {
                    "role": "user",
                    "content": promtp.format(text=section, sdg_definitions=sdg_definitions),
                },
            ],
            temperature=0,
        )